In [ ]:
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import spikeinterface.extractors as se

possible_roots = [
    Path.cwd() / "data",
    Path.cwd().parent / "data",
    Path("data"),
    Path("../data"),
]

data_root = next((root.resolve() for root in possible_roots if root.exists()), None)
if data_root is None:
    raise FileNotFoundError("Could not find the data folder. Expected a 'data' directory near the notebook.")

stream_id = "0"
seconds_per_file = 2.0

folder_to_files = defaultdict(list)
for file_path in sorted(data_root.rglob("*.rhd")):
    folder_to_files[file_path.parent].append(file_path)

if not folder_to_files:
    print(f"No .rhd files found under {data_root}")
else:
    for folder in sorted(folder_to_files):
        files = folder_to_files[folder]
        fig_height = max(2.5, 2.25 * len(files))
        fig, axes = plt.subplots(
            len(files),
            1,
            figsize=(16, fig_height),
            sharex=True,
            constrained_layout=True,
        )

        if len(files) == 1:
            axes = [axes]

        for ax, file_path in zip(axes, files):
            try:
                recording = se.read_intan(str(file_path), stream_id=stream_id)
                sampling_hz = float(recording.get_sampling_frequency())
                n_samples = int(recording.get_num_samples())
                n_channels = int(recording.get_num_channels())
                channel_id = recording.get_channel_ids()[0]
                frame_count = max(1, min(n_samples, int(sampling_hz * seconds_per_file)))
                traces = recording.get_traces(start_frame=0, end_frame=frame_count, channel_ids=[channel_id])
                signal = traces[:, 0]
                time_axis = np.arange(signal.shape[0]) / sampling_hz

                ax.plot(time_axis, signal, linewidth=0.8, color="tab:blue")
                ax.set_title(
                    f"{file_path.name} | {n_channels} ch | {n_samples / sampling_hz:.2f} s",
                    loc="left",
                    fontsize=10,
                )
                ax.set_ylabel("amplitude")
                ax.grid(True, alpha=0.25)
            except Exception as exc:
                ax.text(
                    0.5,
                    0.5,
                    f"Failed to load:\n{file_path.name}\n{exc}",
                    ha="center",
                    va="center",
                    transform=ax.transAxes,
                )
                ax.set_axis_off()

        axes[-1].set_xlabel("time (s)")
        folder_label = folder.relative_to(data_root)
        fig.suptitle(str(folder_label) if str(folder_label) != "." else data_root.name, fontsize=14)
        plt.show()


In [ ]:
# knowledge of mat files
from scipy.io import loadmat
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# knowledge of mat files
mat_root = data_root
mat_files = sorted(mat_root.rglob("*.mat"))

if not mat_files:
    print(f"No .mat files found under {mat_root}")
else:
    def describe_value(value, name="value", indent=0):
        prefix = "  " * indent
        if isinstance(value, np.ndarray):
            print(f"{prefix}{name}: ndarray shape={value.shape}, dtype={value.dtype}")
            if value.dtype.names:
                print(f"{prefix}  fields={value.dtype.names}")
            return
        if isinstance(value, np.void):
            field_names = value.dtype.names or ()
            print(f"{prefix}{name}: struct with fields={field_names}")
            for field_name in field_names:
                describe_value(value[field_name], field_name, indent + 1)
            return
        if isinstance(value, dict):
            print(f"{prefix}{name}: dict with keys={list(value.keys())}")
            for key_name, item in value.items():
                describe_value(item, key_name, indent + 1)
            return
        print(f"{prefix}{name}: {type(value).__name__} -> {value}")

    for file_path in mat_files:
        print("\n" + "=" * 100)
        print(f"FILE: {file_path.relative_to(mat_root)}")
        print("=" * 100)

        mat_data = loadmat(str(file_path), squeeze_me=False, struct_as_record=False)
        data_keys = [key for key in mat_data.keys() if not key.startswith("__")]

        if not data_keys:
            print("No non-metadata variables found in this file.")
            continue

        print(f"Variables: {data_keys}")

        for key in data_keys:
            describe_value(mat_data[key], key)

        for key in data_keys:
            value = mat_data[key]
            if not isinstance(value, np.ndarray):
                continue
            if value.size == 0:
                continue
            if not np.issubdtype(value.dtype, np.number):
                continue

            arr = np.asarray(value)
            print(f"\nPlotting variable '{key}' with shape {arr.shape}")

            if arr.ndim == 1:
                fig, ax = plt.subplots(figsize=(14, 3))
                ax.plot(arr, linewidth=0.8)
                ax.set_title(f"{file_path.name} | {key} | shape={arr.shape}")
                ax.set_xlabel("sample index")
                ax.set_ylabel("value")
                ax.grid(True, alpha=0.25)
                plt.show()
            elif arr.ndim == 2:
                fig, ax = plt.subplots(figsize=(14, 3))
                if arr.shape[1] == 2:
                    ax.plot(arr[:, 0], arr[:, 1], linewidth=0.8)
                    ax.set_xlabel("column 0")
                    ax.set_ylabel("column 1")
                    ax.set_title(f"{file_path.name} | {key} | shape={arr.shape} | 2-column view")
                else:
                    if arr.shape[0] >= arr.shape[1]:
                        y = arr[:, 0]
                    else:
                        y = arr[0, :]
                    ax.plot(y, linewidth=0.8)
                    ax.set_xlabel("sample index")
                    ax.set_ylabel("value")
                    ax.set_title(f"{file_path.name} | {key} | shape={arr.shape} | first vector")
                ax.grid(True, alpha=0.25)
                plt.show()
            else:
                print(f"Skipping plot for '{key}' because it has ndim={arr.ndim} and shape={arr.shape}")

In [ ]:
from scipy.io import loadmat
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

mat_root = data_root
mat_files = sorted(mat_root.rglob("*.mat"))

if not mat_files:
    print(f"No .mat files found under {mat_root}")
else:
    print("Found .mat files:")
    for i, file_path in enumerate(mat_files, start=1):
        print(f"{i}. {file_path.relative_to(mat_root)}")

    # Change this to inspect a different file
    selected_index = 0
    file_path = mat_files[selected_index]

    print("\n" + "=" * 100)
    print(f"INSPECTING: {file_path.relative_to(mat_root)}")
    print("=" * 100)

    mat_data = loadmat(str(file_path), squeeze_me=False, struct_as_record=False)
    data_keys = [key for key in mat_data.keys() if not key.startswith("__")]

    if not data_keys:
        print("No non-metadata variables found in this file.")
    else:
        print(f"Variables: {data_keys}")

        for key in data_keys:
            value = mat_data[key]
            if isinstance(value, np.ndarray):
                print(f"{key}: shape={value.shape}, dtype={value.dtype}")
            else:
                print(f"{key}: type={type(value).__name__}")

        key = data_keys[0]
        data = mat_data[key]

        if not isinstance(data, np.ndarray):
            print(f"\nVariable '{key}' is not an ndarray, so it cannot be plotted directly.")
        elif data.size == 0:
            print(f"\nVariable '{key}' is empty.")
        elif not np.issubdtype(data.dtype, np.number):
            print(f"\nVariable '{key}' is not numeric.")
        else:
            arr = np.asarray(data)

            print(f"\nSelected variable: {key}")
            print(f"Shape: {arr.shape}")
            print(f"Dtype: {arr.dtype}")

            if arr.ndim == 2 and arr.shape[1] == 2:
                col0 = arr[:, 0]
                col1 = arr[:, 1]

                print("\nColumn 0 stats:")
                print(f"  min={np.min(col0):.6g}, max={np.max(col0):.6g}, mean={np.mean(col0):.6g}, std={np.std(col0):.6g}")
                print("Column 1 stats:")
                print(f"  min={np.min(col1):.6g}, max={np.max(col1):.6g}, mean={np.mean(col1):.6g}, std={np.std(col1):.6g}")

                is_time_like = np.all(np.diff(col0) >= 0)
                print(f"\nColumn 0 looks time-like: {is_time_like}")
                if is_time_like:
                    print("Interpretation: column 0 is likely time, column 1 is likely angle/signal.")

                fig, axes = plt.subplots(3, 1, figsize=(14, 10), constrained_layout=True)

                axes[0].plot(col0, linewidth=0.9, color="tab:blue")
                axes[0].set_title(f"{file_path.name} | {key} | column 0")
                axes[0].set_ylabel("col 0")
                axes[0].grid(True, alpha=0.25)

                axes[1].plot(col1, linewidth=0.9, color="tab:green")
                axes[1].set_title(f"{file_path.name} | {key} | column 1")
                axes[1].set_ylabel("col 1")
                axes[1].grid(True, alpha=0.25)

                axes[2].plot(col0, col1, linewidth=0.9, color="tab:purple")
                axes[2].set_title(f"{file_path.name} | {key} | col 0 vs col 1")
                axes[2].set_xlabel("col 0")
                axes[2].set_ylabel("col 1")
                axes[2].grid(True, alpha=0.25)

                plt.show()
            elif arr.ndim == 1:
                print("\n1D array stats:")
                print(f"  min={np.min(arr):.6g}, max={np.max(arr):.6g}, mean={np.mean(arr):.6g}, std={np.std(arr):.6g}")

                fig, ax = plt.subplots(figsize=(14, 3))
                ax.plot(arr, linewidth=0.9, color="tab:blue")
                ax.set_title(f"{file_path.name} | {key} | shape={arr.shape}")
                ax.set_xlabel("sample index")
                ax.set_ylabel("value")
                ax.grid(True, alpha=0.25)
                plt.show()
            else:
                print(f"\nSkipping plot for '{key}' because it has ndim={arr.ndim} and shape={arr.shape}")

from above we gather:
that **column 0 is in seconds.**

Here is the breakdown of the specific evidence from the study ("Classification of naturally evoked compound action potentials in peripheral nerve spatiotemporal recordings") and the data in your image:

### 1. The Frame Rate Calculation
In the **Methods** section of the paper, the authors state that the ankle rotation (the mechanical stimulus) was recorded using a standard Logitech webcam at **30 frames per second (fps)**.

* **Look at your X-axis:** In the top two plots, the horizontal axis goes from 0 to roughly **10,500**. These represent the individual video frames.
* **Look at the Y-axis of the top plot:** At the very end of the recording (frame 10,500), the value of `col 0` is exactly **350**.
* **The Math:** $10,500 \text{ frames} \div 30 \text{ frames per second} = 350 \text{ seconds}$.

This linear relationship ($y = \frac{1}{30}x$) confirms that column 0 is the conversion of the frame count into total elapsed time in seconds.



### 2. The Nature of the Experiment
The experiment involved manually rotating a rat's ankle to evoke neural signals. 
* Doing this for **350 seconds** (about 5 minutes and 50 seconds) is a standard duration for a laboratory recording session to gather enough "trials" for machine learning.
* If the unit were **minutes**, the experiment would have lasted nearly 6 hours ($350 \text{ minutes}$), which is unlikely for this type of acute physiological preparation.
* If the unit were **milliseconds**, the entire recording would have lasted only 0.35 seconds, which is impossible given that the middle plot shows dozens of slow, manual oscillations of the ankle joint.

### 3. The Relationship in the Bottom Plot
The bottom plot is a "Phase Plot" or a time-series plot where `col 1` (Angle) is plotted against `col 0` (Time). 
* The x-axis here goes from **0 to 350**. 
* Because we know the middle plot shows the ankle moving back and forth (60° to -60°), and it takes about 5-8 seconds per full oscillation (which is a natural human hand-movement speed), the 350-second duration fits the physical reality of the experiment perfectly.

**Summary:**
* **col 0:** Time in **seconds**.
* **col 1:** Ankle angle in **degrees**.
* **X-axis (top/mid):** Raw **frame number** from the 30fps video.

In [ ]:
from scipy.io import loadmat
from collections import defaultdict
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

# Find all .mat files in data directory
mat_root = data_root  # Reuse the data_root from previous cell
mat_folder_to_files = defaultdict(list)

for file_path in sorted(mat_root.rglob("*.mat")):
    mat_folder_to_files[file_path.parent].append(file_path)

if not mat_folder_to_files:
    print(f"No .mat files found under {mat_root}")
else:
    for folder in sorted(mat_folder_to_files):
        files = mat_folder_to_files[folder]
        fig_height = max(2.5, 2.25 * len(files))
        fig, axes = plt.subplots(
            len(files),
            1,
            figsize=(16, fig_height),
            sharex=False,
            constrained_layout=True,
        )

        if len(files) == 1:
            axes = [axes]

        for ax, file_path in zip(axes, files):
            try:
                mat_data = loadmat(str(file_path))
                
                # Filter out MATLAB metadata keys
                data_keys = [k for k in mat_data.keys() if not k.startswith('__')]
                
                if not data_keys:
                    ax.text(0.5, 0.5, f"No data arrays in:\n{file_path.name}", 
                           ha="center", va="center", transform=ax.transAxes)
                    ax.set_axis_off()
                    continue
                
                # Use the first data array found
                key = data_keys[0]
                data = mat_data[key]
                
                # Handle 2D arrays (matrix data)
                if data.ndim == 2 and data.shape[1] == 2:
                    # Plot col 0 vs col 1
                    col0 = data[:, 0]
                    col1 = data[:, 1]
                    ax.plot(col0, col1, linewidth=0.8, color="tab:purple")
                    ax.set_xlabel("col 0 (time in seconds)")
                    ax.set_ylabel("col 1 (angle in degrees)")
                elif data.ndim == 2:
                    if data.shape[0] > data.shape[1]:
                        signal = data[:, 0]
                    else:
                        signal = data[0, :]
                    ax.plot(signal, linewidth=0.8, color="tab:green")
                    ax.set_ylabel("amplitude")
                elif data.ndim == 1:
                    ax.plot(data, linewidth=0.8, color="tab:blue")
                    ax.set_ylabel("value")
                else:
                    ax.text(0.5, 0.5, f"Unsupported data shape:\n{data.shape}", 
                           ha="center", va="center", transform=ax.transAxes)
                    ax.set_axis_off()
                    continue
                
                ax.set_title(
                    f"{file_path.name} | shape={data.shape} | key='{key}'",
                    loc="left",
                    fontsize=10,
                )
                ax.grid(True, alpha=0.25)
                
            except Exception as exc:
                ax.text(
                    0.5,
                    0.5,
                    f"Failed to load:\n{file_path.name}\n{type(exc).__name__}: {str(exc)[:50]}",
                    ha="center",
                    va="center",
                    transform=ax.transAxes,
                    fontsize=9,
                )
                ax.set_axis_off()

        folder_label = folder.relative_to(mat_root)
        fig.suptitle(f".mat files: {str(folder_label) if str(folder_label) != '.' else mat_root.name}", fontsize=14)
        plt.show()


In [ ]:
# Overlay .rhd and .mat ensuring they share the same time scale (mat_time in seconds)
from scipy.io import loadmat
import numpy as np
import matplotlib.pyplot as plt

common_folders = sorted([f for f in folder_to_files.keys() if f in mat_folder_to_files])

if not common_folders:
    print("No folders contain both .rhd and .mat files.")
else:
    for folder in common_folders:
        rhd_files = folder_to_files[folder]
        mat_files = mat_folder_to_files[folder]

        fig, axes = plt.subplots(len(rhd_files), 1, figsize=(14, 4 * len(rhd_files)), constrained_layout=True)
        if len(rhd_files) == 1:
            axes = [axes]

        for ax, rhd_file in zip(axes, rhd_files):
            try:
                # Load FULL .rhd file (30 kHz sampling rate) - ALL samples
                recording = se.read_intan(str(rhd_file), stream_id=stream_id)
                sampling_hz = float(recording.get_sampling_frequency())  # 30000 Hz
                n_samples = int(recording.get_num_samples())
                channel_id = recording.get_channel_ids()[0]
                
                # Load entire file
                traces = recording.get_traces(start_frame=0, end_frame=n_samples, channel_ids=[channel_id])
                rhd_signal = traces[:, 0]
                rhd_time = np.arange(rhd_signal.shape[0]) / sampling_hz

                # Load first .mat file for this folder
                mat_data = loadmat(str(mat_files[0]))
                data_keys = [k for k in mat_data.keys() if not k.startswith('__')]
                mat_array = np.asarray(mat_data[data_keys[0]])

                # Expect mat_array[:,0]=time (seconds) and mat_array[:,1]=angle
                if mat_array.ndim >= 2 and mat_array.shape[1] >= 2:
                    mat_time = np.asarray(mat_array[:, 0], dtype=float)
                    mat_angle = np.asarray(mat_array[:, 1], dtype=float)

                    # Use mat_time (col 0) as the x-axis reference (in seconds)
                    # Interpolate rhd_signal onto mat_time scale for alignment
                    rhd_signal_on_mat = np.interp(mat_time, rhd_time, rhd_signal, left=np.nan, right=np.nan)

                    # Plot .rhd and .mat both on mat_time scale (col 0 in seconds)
                    ax2 = ax.twinx()
                    ax.plot(mat_time, rhd_signal_on_mat, linewidth=0.5, color="tab:blue", label=".rhd signal", alpha=0.8)
                    ax2.plot(mat_time, mat_angle, linewidth=1.0, color="tab:orange", label=".mat angle")

                    ax.set_xlabel("time (s) [from mat file col 0]")
                    ax.set_ylabel(".rhd amplitude", color="tab:blue")
                    ax2.set_ylabel(".mat angle (deg)", color="tab:orange")

                    # Optional: show a small legend
                    lines, labels = ax.get_legend_handles_labels()
                    lines2, labels2 = ax2.get_legend_handles_labels()
                    ax.legend(lines + lines2, labels + labels2, loc="upper right", fontsize=8)
                else:
                    # Fallback: plot first column/rows as before
                    data = mat_array
                    if data.ndim == 2 and data.shape[0] > data.shape[1]:
                        signal = data[:, 0]
                    else:
                        signal = data[0, :]
                    ax.plot(signal, linewidth=0.8, color="tab:green")
                    ax.set_ylabel("amplitude")

                ax.set_title(f"{rhd_file.name} (full {n_samples/sampling_hz:.1f}s)")
                ax.grid(True, alpha=0.25)

            except Exception as exc:
                ax.text(0.5, 0.5, f"Failed to load:\n{rhd_file.name}\n{type(exc).__name__}: {str(exc)[:200]}",
                       ha="center", va="center", transform=ax.transAxes)
                ax.set_axis_off()

        fig.suptitle(f"{folder.name} - .rhd vs .mat overlay (full RHD file, time scale: mat file col 0)", fontsize=14)
        plt.show()